# M18 · LLM fundamentals + prompting

AFP-AI · Domain 4 · LLMs

**Turn tokens into probabilities and prompts into reliable structured behavior.**

We simulate a tiny language model with NumPy. The core probability is

$$P(x_1,\ldots,x_T)=\prod_t P(x_t\mid x_{<t})$$

and perplexity is $\exp$ of average negative log probability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(18)

## A toy tokenizer

Real tokenizers split text into subwords. Here we lowercase and split on spaces so we can focus on probability mechanics.

In [ ]:
vocab = ["return", "relevant", "broad", "unsafe", "json", "fallback"]
token_to_id = {token: idx for idx, token in enumerate(vocab)}

def tokenize(text):
    cleaned = text.lower().replace(",", "")
    return cleaned.split()

prompt = "Return json fallback"
tokens = tokenize(prompt)
ids = [token_to_id[token] for token in tokens]

print(tokens)
print(ids)

assert ids == [0, 4, 5]

## Softmax and temperature

A model emits logits. Temperature computes $\text{softmax}(z/\tau)$, so smaller $\tau$ makes the largest logit more dominant.

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits)
    exp_logits = np.exp(shifted)
    return exp_logits / exp_logits.sum()

logits = np.array([0.2, 2.0, 1.0, 0.0, -0.5, -1.0])
probs_t1 = softmax(logits / 1.0)
probs_t05 = softmax(logits / 0.5)
probs_t2 = softmax(logits / 2.0)

print(np.round(probs_t1, 3))
print(np.round(probs_t05, 3))
print(np.round(probs_t2, 3))

assert probs_t05[1] > probs_t1[1]

## Sequence probability and perplexity

If a target answer has token probabilities $0.7$, $0.5$, and $0.25$, the sequence probability is their product. Perplexity converts average log loss back to an intuitive effective branching factor.

In [ ]:
target_probs = np.array([0.7, 0.5, 0.25])
sequence_probability = np.prod(target_probs)
negative_log_likelihood = -np.sum(np.log(target_probs))
perplexity = np.exp(negative_log_likelihood / len(target_probs))

print("sequence probability", round(sequence_probability, 4))
print("perplexity", round(perplexity, 3))

assert np.isclose(sequence_probability, 0.0875)

## Tiny structured-output scorer

We simulate prompting by adding a bias to logits when the prompt asks for JSON. This is not a real LLM; it is a transparent way to see how instructions can make structured tokens more likely.

In [ ]:
base_logits = np.array([0.1, 1.5, 1.0, 0.2, 0.0, -0.5])
json_bias = np.array([0.0, 0.0, 0.0, 0.0, 2.0, 1.0])
plain_probs = softmax(base_logits)
structured_probs = softmax(base_logits + json_bias)

summary = pd.DataFrame({
    "token": vocab,
    "plain": plain_probs,
    "structured": structured_probs,
})

print(summary.round(3))

assert structured_probs[token_to_id["json"]] > plain_probs[token_to_id["json"]]

## Top-p filtering

Top-p sampling keeps the smallest set of tokens whose cumulative probability reaches $p$. It trims the long tail while preserving some variety.

In [ ]:
def top_p_keep(probs, p):
    order = np.argsort(probs)[::-1]
    cumulative = np.cumsum(probs[order])
    keep_sorted = cumulative <= p
    keep_sorted[0] = True
    first_over = np.argmax(cumulative >= p)
    keep_sorted[:first_over + 1] = True
    keep = np.zeros_like(probs, dtype=bool)
    keep[order[keep_sorted]] = True
    return keep

keep = top_p_keep(probs_t1, 0.8)

print([token for token, flag in zip(vocab, keep) if flag])

assert keep.sum() >= 1

## Visualize temperature

The same logits become sharper or flatter as temperature changes.

In [ ]:
x = np.arange(len(vocab))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, probs_t05, marker="o", label="tau=0.5")
ax.plot(x, probs_t1, marker="o", label="tau=1")
ax.plot(x, probs_t2, marker="o", label="tau=2")
ax.set_xticks(x)
ax.set_xticklabels(vocab, rotation=25)
ax.set_ylabel("probability")
ax.set_title("temperature changes next-token probabilities")
ax.legend()
plt.show()

## Practice

1. Change the JSON bias and observe when `fallback` becomes more likely than `json`.
2. Try top-p values 0.5, 0.8, and 0.95.
3. Create a three-example few-shot prompt as a Python list of dictionaries and count its tokens.

In [ ]:
# Your turn:
